In [1]:
import json
from pathlib import Path
import pandas as pd
import re
import extract as ext
import pdfplumber

In [2]:
ROOT = Path("~/code/selector_examenes").expanduser()

In [3]:
def load_keywords(json_file=ROOT / "json/quimica_keywords.json"):
    with open(json_file) as file:
        return json.load(file)

In [4]:
def extract_statement(exercise):
    m = re.search("R E S O L U C I Ó N", exercise)
    if m:
        end_pos = m.start()
        exercise = exercise[:end_pos].strip()
        lines = exercise.split("\n")
        exam = lines[-1]
        statement = "\n".join(lines[:-1])
        return exam, statement
    return None

In [5]:
def extract_exercises(file):
    exercises = []
    with pdfplumber.open(file) as pdf:
        for idx, page in enumerate(pdf.pages):
            texto = ext.extract_text_from_page(page)
            if idx > 0:  # ejercicios comienzan en 2da página
                st = extract_statement(texto)
                if st:
                    exercises.append(extract_statement(texto))
                else:
                    print(f"Error extrayendo ejercicios de archivo: {file.name}, página {idx+1}")
                    with open("errores.txt", "a") as f:
                        f.write(texto)
                        
    return exercises

In [6]:
def classify_exercise(statement: str, topic: str, keywords = None) -> str:
    if keywords is None:
        keywords = load_keywords()

    # convertir saltos de línea en espacios
    statement = re.sub(r"\s+", " " , statement)

    
    # Vocales acentuadas (minúsculas y mayúsculas) → sin acentuar
    tabla = str.maketrans(
        "áéíóúàèìòùäëïöüâêîôûÁÉÍÓÚÀÈÌÒÙÄËÏÖÜÂÊÎÔÛ",
        "aeiouaeiouaeiouaeiouAEIOUAEIOUAEIOUAEIOU"
        )

    statement = statement.translate(tabla)
    topic_keywords = keywords[topic]["topic_kw"]
    pattern = re.compile("|".join(topic_keywords), flags=re.IGNORECASE | re.DOTALL)
    if re.search(pattern, statement):
        return topic

In [ ]:
def create_df_exercises(folder: Path) -> pd.DataFrame:
    exercises_list = []
    if not isinstance(folder, Path):
        raise ValueError("La función espera un objeto Path con el directorio a procesar.")
        if not folder.is_dir():
            raise ValueError("El objeto Path debe ser un directorio.")
    
    for file in folder.iterdir():
        if not file.suffix == ".pdf": 
            continue

        exercises = extract_exercises(file)
        for ex in exercises:
            if ex:
                exercises_list.append({"exam" : ex[0], "statement": ex[1]})
    return pd.DataFrame(exercises_list)

Ordenar directorios sin acentos:

In [12]:
TEMAS = sorted([i for i in Path("quimica_temas").iterdir() if i.is_dir()])

In [13]:
for idx, topic in enumerate([i.name for i in TEMAS]):
    print(f"{idx}: {topic}")

0: acido_base
1: cantidad_quimica
2: configuracion_electronica
3: enlace_quimico
4: equilibrio_quimico
5: formulacion
6: reactividad_organica
7: redox
8: solubilidad
9: termoquimica


## Clasificar carpeta de ejercicios

In [22]:
success_strings = []
for t in TEMAS:

    df = create_df_exercises(t)
    df["topic"] = df.statement.apply(classify_exercise, topic=t.name)
    df = df.explode("topic")
    df = df.sort_values("exam").reset_index(drop=True)
    df = df.drop_duplicates()

    not_classified_count = df.topic.isna().sum()
    success_strings.append(f"Tema: {t.name}: {round( (df.shape[0] - not_classified_count) / df.shape[0] * 100, 1)} % aciertos.")

Error extrayendo ejercicios de archivo: 2009 - Ácido Base.pdf, página 3
Error extrayendo ejercicios de archivo: 2025 - Ácido Base.pdf, página 5
Error extrayendo ejercicios de archivo: 2026 - Ácido Base.pdf, página 5
Error extrayendo ejercicios de archivo: 2026 - Enlace Químico.pdf, página 6
Error extrayendo ejercicios de archivo: 2021 - Enlace Químico.pdf, página 3
Error extrayendo ejercicios de archivo: 2026 - Reacciones Redox.pdf, página 3
Error extrayendo ejercicios de archivo: 2026 - Reacciones Redox.pdf, página 9
Error extrayendo ejercicios de archivo: 2025 - Reacciones Redox.pdf, página 3
Error extrayendo ejercicios de archivo: 2026 - Solubilidad.pdf, página 4



Tema: acido_base: 99.2 % aciertos.
Tema: cantidad_quimica: 51.0 % aciertos.
Tema: configuracion_electronica: 92.0 % aciertos.
Tema: enlace_quimico: 93.6 % aciertos.
Tema: equilibrio_quimico: 86.4 % aciertos.
Tema: formulacion: 100.0 % aciertos.
Tema: reactividad_organica: 86.2 % aciertos.
Tema: redox: 96.1 % acierto

In [23]:
print(*success_strings, sep="\n")

Tema: acido_base: 99.2 % aciertos.
Tema: cantidad_quimica: 51.0 % aciertos.
Tema: configuracion_electronica: 92.0 % aciertos.
Tema: enlace_quimico: 93.6 % aciertos.
Tema: equilibrio_quimico: 86.4 % aciertos.
Tema: formulacion: 100.0 % aciertos.
Tema: reactividad_organica: 86.2 % aciertos.
Tema: redox: 96.1 % aciertos.
Tema: solubilidad: 97.7 % aciertos.
Tema: termoquimica: 96.9 % aciertos.
